In [1]:
print("yes, it is 08 nxxext metrics")

yes, it is 08 nxxext metrics


In [2]:
%cd ~

/root


In [3]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme

Cloning into 'Dual_watermarking_Scheme'...
remote: Enumerating objects: 242, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 242 (delta 47), reused 55 (delta 16), pack-reused 151 (from 1)
Receiving objects: 100% (242/242), 43.56 MiB | 20.06 MiB/s, done.
Resolving deltas: 100% (106/106), done.


In [4]:
!ls

Dual_watermarking_Scheme


In [5]:
%cd Dual_watermarking_Scheme/

/root/Dual_watermarking_Scheme


In [6]:
!ls

choices.md  model      PLAN.md	  requirements.txt  src
data	    notebooks  README.md  setup.md


In [7]:
from huggingface_hub import login
login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [8]:
import math
from collections import Counter

import numpy as np
import torch
import pandas as pd
from scipy.stats import binomtest
from transformers import AutoModelForCausalLM, AutoTokenizer


from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

from datasets import load_dataset

from src.utils.key_manager import derive_set


# ============================================================
# Judge model loading (independent of the watermark generator)
# ============================================================
_JUDGE_CACHE = {}


def load_judge_model(judge_model_name="gpt2-large", device="cuda"):
    """
    Loads (and caches) an independent judge model for external perplexity
    scoring. Default: gpt2-large. Swap to e.g. "EleutherAI/pythia-1.4b" if
    preferred. NEVER pass an OPT checkpoint here - that would bias the
    eval toward the watermark's own generator.
    """
    if judge_model_name in _JUDGE_CACHE:
        return _JUDGE_CACHE[judge_model_name]

    tokenizer = AutoTokenizer.from_pretrained(judge_model_name)
    model = AutoModelForCausalLM.from_pretrained(judge_model_name).to(device)
    model.eval()

    _JUDGE_CACHE[judge_model_name] = (model, tokenizer)
    return model, tokenizer


def split_continuation(prompt, full_text):
    """
    Your generation pipeline decodes: prompt + generated continuation.
    This extracts only the generated continuation, so metrics aren't
    diluted by the (identical, uninteresting) prompt text.
    Falls back to the full text if the prefix doesn't match exactly
    (can happen due to tokenizer detokenization quirks).
    """
    if full_text.startswith(prompt):
        return full_text[len(prompt):].strip()
    return full_text.strip()


@torch.no_grad()
def _perplexity_core(prompt, continuation, model, tokenizer, device="cuda"):
    """
    Shared implementation: perplexity of `continuation` under `model`,
    conditioned on `prompt` as context. Only continuation tokens
    contribute to the loss (prompt tokens are masked with -100).
    PPL = exp(average negative log likelihood).
    """
    if not continuation.strip():
        return float("nan")

    prompt_enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    full_enc = tokenizer(prompt + continuation, return_tensors="pt", add_special_tokens=False)

    prompt_ids = prompt_enc.input_ids
    full_ids = full_enc.input_ids
    prompt_len = prompt_ids.shape[1]

    max_len = getattr(model.config, "max_position_embeddings", None)
    if max_len is None:
        max_len = getattr(model.config, "n_positions", 1024)

    if full_ids.shape[1] > max_len:
        full_ids = full_ids[:, :max_len]

    prompt_len = min(prompt_len, full_ids.shape[1])
    full_ids = full_ids.to(device)

    labels = full_ids.clone()
    labels[:, :prompt_len] = -100  # mask prompt tokens out of the loss

    if (labels != -100).sum().item() == 0:
        return float("nan")  # prompt ate the whole context window

    outputs = model(input_ids=full_ids, labels=labels)
    return torch.exp(outputs.loss).item()


def compute_self_perplexity(prompt, continuation, model, tokenizer, device="cuda"):
    """
    Perplexity scored by the SAME model that generated the text
    (e.g. OPT-2.7B generates -> OPT-2.7B evaluates). Useful as a sanity
    check, but NOT the required quality metric - use compute_external_perplexity
    for the reported numbers, since a model scoring its own text is biased
    toward looking fluent.
    """
    return _perplexity_core(prompt, continuation, model, tokenizer, device)


def compute_external_perplexity(prompt, continuation, judge_model, judge_tokenizer, device="cuda"):
    """
    Perplexity scored by an INDEPENDENT judge model
    (e.g. OPT-2.7B generates -> GPT-2-large evaluates). This is the
    metric required by the spec.
    """
    return _perplexity_core(prompt, continuation, judge_model, judge_tokenizer, device)


def compute_self_perplexity_batch(prompts, continuations, model, tokenizer, device="cuda"):
    return [
        compute_self_perplexity(p, c, model, tokenizer, device)
        for p, c in zip(prompts, continuations)
    ]


def compute_external_perplexity_batch(prompts, continuations, judge_model, judge_tokenizer, device="cuda"):
    return [
        compute_external_perplexity(p, c, judge_model, judge_tokenizer, device)
        for p, c in zip(prompts, continuations)
    ]


In [ ]:
from src.watermark.dual_layer import DualWaterMarking
from src.metrics.layer2 import (load_judge_model,split_continuation,
                                compute_self_perplexity_batch,compute_external_perplexity_batch)

In [10]:
!ls

choices.md  model      PLAN.md	  requirements.txt  src
data	    notebooks  README.md  setup.md


In [11]:
c4_datapath = "data/extracted/c4_samples.parquet"

c4_df = pd.read_parquet(c4_datapath)

In [12]:
c4_df

,prompt_text,prompt_tokens,reference_text,reference_tokens,full_text,domain
0,"""Whoever gets him, they'll be getting a good o...","[2, 113, 39858, 1516, 123, 6, 51, 581, 28, 562...",\nIt’s an experience that might humble some. B...,"[50118, 243, 17, 27, 29, 41, 676, 14, 429, 140...","""Whoever gets him, they'll be getting a good o...",realnews
1,8i announced today it has launched a web playe...,"[2, 398, 118, 585, 452, 24, 34, 1660, 10, 3748...",walk around a person inside a virtual environ...,"[1656, 198, 10, 621, 1025, 10, 6229, 1737, 7, ...",8i announced today it has launched a web playe...,realnews
2,The Arlington County Board plans to vote Satur...,"[2, 133, 15728, 413, 1785, 708, 7, 900, 378, 1...","member board is expected to support the plan, ...","[8648, 792, 16, 421, 7, 323, 5, 563, 6, 61, 21...",The Arlington County Board plans to vote Satur...,realnews
3,The Hawaii man who was fired after issuing the...,"[2, 133, 6467, 313, 54, 21, 2277, 71, 10392, 5...",former state employee – a man in his 50s who ...,"[320, 194, 3200, 126, 10, 313, 11, 39, 654, 29...",The Hawaii man who was fired after issuing the...,realnews
4,Shania Twain expected to break the charts with...,"[2, 3609, 8295, 34878, 421, 7, 1108, 5, 12201,...",song LP was released on Sept. 29 and is set to...,"[17481, 8765, 21, 703, 15, 1919, 4, 1132, 8, 1...",Shania Twain expected to break the charts with...,realnews
...,...,...,...,...,...,...
995,If you’re thinking of buying property in Costa...,"[2, 1106, 47, 17, 27, 241, 2053, 9, 2159, 1038...",among the benefits they provide. Condos also ...,"[566, 5, 1795, 51, 694, 4, 12108, 366, 67, 492...",If you’re thinking of buying property in Costa...,realnews
996,KARACHI - Gunmen shot dead a young man while l...,"[2, 530, 2747, 11083, 100, 111, 6409, 2262, 73...",imar area of Pak Colony police jurisdiction. P...,"[34599, 443, 9, 15776, 27904, 249, 10542, 4, 5...",KARACHI - Gunmen shot dead a young man while l...,realnews
997,The fake Chamber press release distributed to ...,"[2, 133, 4486, 7514, 1228, 800, 7664, 7, 1865,...","misspelled, and the individuals listed as pre...","[2649, 30514, 6, 8, 5, 2172, 3147, 25, 1228, 9...",The fake Chamber press release distributed to ...,realnews
998,One of Iowa’s only aquaponic farms is now prov...,"[2, 3762, 9, 4109, 17, 27, 29, 129, 16690, 369...",is president and CEO of the facility. Moulds ...,"[16, 394, 8, 1324, 9, 5, 2122, 4, 256, 25513, ...",One of Iowa’s only aquaponic farms is now prov...,realnews


In [13]:
prompts = c4_df["prompt_text"].tolist()

prompts = prompts[:100]
print(prompts[:5])


['"Whoever gets him, they\'ll be getting a good one," David Montgomery said.\nINDIANAPOLIS — Hakeem Butler has been surrounded by some of the best wide receivers on the planet this week at the NFL Scouting Combine.', '8i announced today it has launched a web player, the 8i Portal, for its volumetric 3D video of people in virtual reality.\nUsing 8i’s technology and VR goggles, you’ll be able to', 'The Arlington County Board plans to vote Saturday afternoon on giving Amazon $23 million and other incentives to build a headquarters campus in Crystal City, but only after hearing scores of northern Virginia residents and advocates testify for or against the project.\nThe five-', 'The Hawaii man who was fired after issuing the false ballistic missile alert in mid-January told reporters Friday that he was very upset over the incident but remained adamant that it appeared, at the time, to be a real-life attack.\nThe', 'Shania Twain expected to break the charts with new album NOW!\nEven after a 

In [14]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [15]:
from huggingface_hub import login
login()

import torch
from src.utils.model import load_model
from src.utils.loadConfig import load_config
from src.utils.key_manager import generate_key

config = load_config("secondLayer.json")
device = "cuda" if torch.cuda.is_available() else "cpu"
model,tokenizer,vocab_size = load_model(config["MODEL_NAME"])
key = generate_key()
model.to(device)

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.30GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model and tokenizer of facebook/opt-2.7b loaded with vocab_size 50265


OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 2560, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 2560)
      (final_layer_norm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-31): 32 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
            (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
            (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
            (out_proj): Linear(in_features=2560, out_features=2560, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear(in_features=10240, out_features=2560, bias=True)
          (final_layer_norm): Laye

In [16]:
!ls data/greenlist

entertainment.csv  history.csv	 politics.csv  sports.csv
finance.csv	   medicine.csv  science.csv   technology.csv


In [17]:
key = generate_key()
key

'a73fa5e52d5826ead25a066e29b38d61d8924bd40069d76699d5e920e7e5bd85'

In [18]:
!ls

choices.md  model      PLAN.md	  requirements.txt  src
data	    notebooks  README.md  setup.md


In [102]:
dual = DualWaterMarking(
    model=model,
    tokenizer=tokenizer,
    key=key,
    greenlist_dir="data/greenlist",
    max_new_tokens=50,
    temperature=1.0,
    top_p=0.9
)

In [20]:
dual = DualWaterMarking(
    model=model,
    tokenizer=tokenizer,
    key=key,
    greenlist_dir="data/greenlist",
    max_new_tokens=50,
    temperature=1.0,
    top_p=0.9
)

# keep cosine similarity in float32
dual.topic_matrix = dual.topic_matrix.to(
    device=model.device,
    dtype=torch.float32
)

dual.normed_embeddings = dual.normed_embeddings.to(
    device=model.device,
    dtype=torch.float32
)

c4_df = pd.read_parquet(
    "data/extracted/c4_samples.parquet"
)

prompts = c4_df["prompt_text"].tolist()[:10]

results = dual.watermark(
    prompts,
    include_single_layers=True
)

In [21]:
row = results.iloc[4]

print("PROMPT:")
print(row["prompt"])

print("\nPLAIN OUTPUT:")
print(row["plain_output"])

print("\nDUAL WATERMARK OUTPUT:")
print(row["dual_watermarked_output"])

print("\nLAYER 1 ONLY:")
print(row["layer1_only_output"])

print("\nLAYER 2 ONLY:")
print(row["layer2_only_output"])

PROMPT:
Shania Twain expected to break the charts with new album NOW!
Even after a 15-year hiatus, she’s still the one! Shania Twain is on pace to top the charts with her new album NOW. The 16-

PLAIN OUTPUT:
Shania Twain expected to break the charts with new album NOW!
Even after a 15-year hiatus, she’s still the one! Shania Twain is on pace to top the charts with her new album NOW. The 16-track LP is already her biggest album since 1999's Up! — and that was just when she had her own No. 1 song, "You're Still The One." The album is her first since her 2002 album, Up! which hit No.

DUAL WATERMARK OUTPUT:
Shania Twain expected to break the charts with new album NOW!
Even after a 15-year hiatus, she’s still the one! Shania Twain is on pace to top the charts with her new album NOW. The 16-track CD features collaborations with Kendrick Lamar and Gwen Stefani. She says the current focus is on music and releasing her new LP, which comes out Friday. She’s been working on this LP for more tha

In [22]:
# ============================================================
# Perplexity evaluation for Dual Watermarking
# Self PPL + External PPL
# ============================================================

device = "cuda"


def get_continuations(df, column):
    return [
        split_continuation(prompt, output)
        for prompt, output in zip(
            df["prompt"],
            df[column]
        )
    ]


# ------------------------------------------------------------
# 1. SELF PERPLEXITY
# OPT-2.7B evaluates its own generations
# ------------------------------------------------------------

results["plain_self_ppl"] = compute_self_perplexity_batch(
    results["prompt"].tolist(),
    get_continuations(results, "plain_output"),
    model,
    tokenizer,
    device
)


results["dual_self_ppl"] = compute_self_perplexity_batch(
    results["prompt"].tolist(),
    get_continuations(results, "dual_watermarked_output"),
    model,
    tokenizer,
    device
)


results["layer1_self_ppl"] = compute_self_perplexity_batch(
    results["prompt"].tolist(),
    get_continuations(results, "layer1_only_output"),
    model,
    tokenizer,
    device
)


results["layer2_self_ppl"] = compute_self_perplexity_batch(
    results["prompt"].tolist(),
    get_continuations(results, "layer2_only_output"),
    model,
    tokenizer,
    device
)



# ------------------------------------------------------------
# 2. EXTERNAL PERPLEXITY
# GPT-2-large evaluates OPT-2.7B generations
# ------------------------------------------------------------

judge_model, judge_tokenizer = load_judge_model(
    "gpt2-large",
    device="cuda"
)


results["plain_external_ppl"] = compute_external_perplexity_batch(
    results["prompt"].tolist(),
    get_continuations(results, "plain_output"),
    judge_model,
    judge_tokenizer,
    device
)


results["dual_external_ppl"] = compute_external_perplexity_batch(
    results["prompt"].tolist(),
    get_continuations(results, "dual_watermarked_output"),
    judge_model,
    judge_tokenizer,
    device
)


results["layer1_external_ppl"] = compute_external_perplexity_batch(
    results["prompt"].tolist(),
    get_continuations(results, "layer1_only_output"),
    judge_model,
    judge_tokenizer,
    device
)


results["layer2_external_ppl"] = compute_external_perplexity_batch(
    results["prompt"].tolist(),
    get_continuations(results, "layer2_only_output"),
    judge_model,
    judge_tokenizer,
    device
)



# ------------------------------------------------------------
# 3. Display results
# ------------------------------------------------------------

print(
    results[
        [
            "topic",

            "plain_self_ppl",
            "dual_self_ppl",
            "layer1_self_ppl",
            "layer2_self_ppl",

            "plain_external_ppl",
            "dual_external_ppl",
            "layer1_external_ppl",
            "layer2_external_ppl"
        ]
    ].head()
)


# ------------------------------------------------------------
# 4. Mean PPL
# ------------------------------------------------------------

print("\n===== Average Perplexity =====")

print(
    results[
        [
            "plain_self_ppl",
            "dual_self_ppl",
            "layer1_self_ppl",
            "layer2_self_ppl",

            "plain_external_ppl",
            "dual_external_ppl",
            "layer1_external_ppl",
            "layer2_external_ppl"
        ]
    ].mean()
)


# ------------------------------------------------------------
# 5. Save
# ------------------------------------------------------------

results.to_csv(
    "c4_dual_watermark_perplexity_results.csv",
    index=False
)

print("\nSaved: c4_dual_watermark_perplexity_results.csv")

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


        topic  plain_self_ppl  dual_self_ppl  layer1_self_ppl  \
0     history        7.169319      11.062344         8.327236   
1  technology        8.545604       5.122869         9.277987   
2     history        4.882146      15.319068         5.323348   
3     history        5.389591       7.701670         6.366690   
4     history        7.052059       8.704086         7.321252   

   layer2_self_ppl  plain_external_ppl  dual_external_ppl  \
0         4.186942           10.389614          20.868242   
1         9.987986           13.570463           8.682527   
2         3.909018            6.994500          23.234632   
3         5.726242           11.703853          21.345100   
4         3.207310           13.222692          15.412642   

   layer1_external_ppl  layer2_external_ppl  
0            38.018024             5.800967  
1            13.698254            13.323483  
2             7.692480             5.440747  
3            17.852005            10.468304  
4           